# 03 — Evaluación del Modelo y Validación del Sistema
**HackIAthon 2026 | Reto Aseguradora del Sur — FraudIA Claims**

Este notebook evalúa el desempeño del modelo ML, valida las reglas de negocio y analiza la explicabilidad del score híbrido.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import sys
from pathlib import Path

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')

# Paths
DATA_PATH  = Path('../backend/ai_data_core/data/synthetic/siniestros_scored.csv')
MODEL_PATH = Path('../backend/ai_data_core/data/processed/fraud_model.pkl')
ENC_PATH   = Path('../backend/ai_data_core/data/processed/label_encoders.pkl')
BACKEND    = Path('../backend')
sys.path.insert(0, str(BACKEND))

df = pd.read_csv(DATA_PATH)
print(f'Dataset: {df.shape[0]} filas | Fraudes simulados: {df["etiqueta_fraude_simulada"].sum()}')

## 1. Cargar Modelo y Preparar Test Set

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    modelo = pickle.load(f)
with open(ENC_PATH, 'rb') as f:
    encoders = pickle.load(f)
print('✅ Modelo cargado:', type(modelo).__name__)

FEATURES_NUM = [
    'monto_reclamado', 'monto_estimado',
    'dias_desde_inicio_poliza', 'dias_desde_fin_poliza',
    'dias_entre_ocurrencia_reporte', 'historial_siniestros_asegurado',
    'score_riesgo', 'tiene_doc_inconsistente',
]
FEATURES_CAT = ['ramo', 'cobertura', 'estado', 'sucursal']
TARGET = 'etiqueta_fraude_simulada'

df_ml = df.copy()
df_ml['suma_asegurada_proxy'] = df_ml['monto_reclamado'] / (df_ml['monto_estimado'] + 1)
df_ml['es_borde_vigencia']    = (df_ml['dias_desde_inicio_poliza'] <= 30).astype(int)
df_ml['reporte_tardio']       = (df_ml['dias_entre_ocurrencia_reporte'] > 7).astype(int)
df_ml['tiene_doc_inconsistente'] = df_ml['tiene_doc_inconsistente'].map(
    {True: 1, False: 0, 1: 1, 0: 0}).fillna(0).astype(int)

for col in FEATURES_CAT:
    le = encoders.get(col, LabelEncoder())
    df_ml[col + '_enc'] = le.transform(df_ml[col].astype(str).fillna('desconocido'))

FEATURES_FINAL = FEATURES_NUM + ['suma_asegurada_proxy', 'es_borde_vigencia', 'reporte_tardio'] + \
                 [c + '_enc' for c in FEATURES_CAT]

X = df_ml[FEATURES_FINAL].fillna(0)
y = df_ml[TARGET].astype(int)

_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f'Test set: {X_test.shape[0]} muestras | Fraudes: {y_test.sum()}')

## 2. Métricas de Clasificación

In [ ]:
y_pred      = modelo.predict(X_test)
y_pred_prob = modelo.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred, zero_division=0)
recall    = recall_score(y_test, y_pred, zero_division=0)
f1        = f1_score(y_test, y_pred, zero_division=0)
auc_roc   = roc_auc_score(y_test, y_pred_prob)
avg_prec  = average_precision_score(y_test, y_pred_prob)

print('=' * 50)
print('MÉTRICAS DEL MODELO EN TEST SET')
print('=' * 50)
print(f'  Precision:        {precision:.3f}')
print(f'  Recall:           {recall:.3f}')
print(f'  F1-Score:         {f1:.3f}')
print(f'  AUC-ROC:          {auc_roc:.3f}')
print(f'  Avg Precision:    {avg_prec:.3f}')
print('=' * 50)
print('\nReporte completo:')
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude']))

## 3. Curvas ROC y Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
axes[0].plot(fpr, tpr, color='#6366f1', lw=2, label=f'AUC-ROC = {auc_roc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Clasificador aleatorio')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#6366f1')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('Tasa de Falsos Positivos')
axes[0].set_ylabel('Tasa de Verdaderos Positivos')
axes[0].set_title('Curva ROC')
axes[0].legend()

# Precision-Recall
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_pred_prob)
axes[1].plot(rec_curve, prec_curve, color='#f59e0b', lw=2, label=f'AP = {avg_prec:.3f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall')
axes[1].legend()

plt.tight_layout()
plt.savefig('curvas_evaluacion.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Normal', 'Fraude'],
    yticklabels=['Normal', 'Fraude']
)
plt.title('Matriz de Confusión')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Verdaderos Negativos (TN): {tn}  — Casos normales correctamente clasificados')
print(f'Falsos Positivos     (FP): {fp}  — Casos normales marcados como fraude (revisión innecesaria)')
print(f'Falsos Negativos     (FN): {fn}  — Fraudes NO detectados (riesgo operativo)')
print(f'Verdaderos Positivos (TP): {tp}  — Fraudes detectados correctamente')

## 5. Validación del Motor de Reglas

In [ ]:
from src.engine.rules_evaluator import ReglasNegocioEngine

# Probar las reglas con casos de ejemplo
casos_prueba = [
    {
        'nombre': 'Caso VERDE (bajo riesgo)',
        'siniestro': {
            'dias_desde_inicio_poliza': 180,
            'dias_desde_fin_poliza': 180,
            'cobertura': 'Choque',
            'dias_entre_ocurrencia_reporte': 1,
            'historial_siniestros_asegurado': 0,
            'documentos_completos': True,
            'tiene_doc_inconsistente': False,
            'monto_reclamado': 3000,
            'descripcion': 'Colisión con otro vehículo.',
        }
    },
    {
        'nombre': 'Caso ROJO (borde vigencia + lista restrictiva + docs inconsistentes)',
        'siniestro': {
            'dias_desde_inicio_poliza': 5,
            'dias_desde_fin_poliza': 5,
            'cobertura': 'Robo',
            'dias_entre_ocurrencia_reporte': 8,
            'historial_siniestros_asegurado': 4,
            'documentos_completos': False,
            'tiene_doc_inconsistente': True,
            'monto_reclamado': 45000,
            'descripcion': 'Vehículo robado sin tercero identificado.',
        },
        'proveedor': {'en_lista_restrictiva': True, 'pct_casos_observados': 0.6}
    },
]

for caso in casos_prueba:
    motor = ReglasNegocioEngine(
        siniestro=caso['siniestro'],
        proveedor=caso.get('proveedor')
    )
    resultado = motor.ejecutar_motor()
    print(f'\n{'='*60}')
    print(f'🔍 {caso["nombre"]}')
    print(f'   Score: {resultado["score_normalizado"]} | Nivel: {resultado["nivel_riesgo"]}')
    print(f'   Alertas activadas ({resultado["total_alertas"]}):')
    for alerta in resultado['alertas_activadas'].split(' | '):
        print(f'     • {alerta}')
    if resultado['reglas_criticas']:
        print(f'   Reglas críticas: {resultado["reglas_criticas"]}')

## 6. Score Híbrido — Comparación Reglas vs ML vs Híbrido

In [ ]:
from src.engine.risk_scorer import calcular_score_hibrido

# Muestra de 20 siniestros al azar
muestra = df.sample(20, random_state=42).to_dict('records')

comparacion = []
for sin in muestra:
    res = calcular_score_hibrido(siniestro=sin)
    comparacion.append({
        'id':           sin['id_siniestro'],
        'score_reglas': res['score_reglas'],
        'score_bd':     sin['score_riesgo'],
        'nivel_calc':   res['nivel_riesgo'],
        'nivel_bd':     sin['nivel_riesgo'],
        'alertas':      res['total_alertas'],
    })

df_comp = pd.DataFrame(comparacion)
coincidencias = (df_comp['nivel_calc'] == df_comp['nivel_bd']).sum()
print(f'Coincidencias nivel riesgo (reglas vs BD): {coincidencias}/20 ({coincidencias*5}%)')
df_comp

## 7. Resumen de Métricas para el Jurado

| Métrica | Valor |
|---|---|
| Precision | Ver celda 2 |
| Recall | Ver celda 2 |
| F1-Score | Ver celda 2 |
| AUC-ROC | Ver celda 2 |
| Señales implementadas | 13 señales + 7 reglas críticas |
| Enfoque | Híbrido: Reglas (60%) + ML (40%) |
| Modelo ML | XGBoost / RandomForest |
| Agente conversacional | Google Gemini 2.0 Flash Lite |

## 8. Limitaciones y Consideraciones Éticas

1. **Datos sintéticos**: El modelo fue entrenado con datos generados artificialmente. El rendimiento real puede variar con datos históricos de producción.
2. **Falsos positivos**: Algunos casos legítimos pueden ser marcados como sospechosos. La revisión humana es **obligatoria** antes de cualquier decisión.
3. **Sesgo**: Las variables categóricas (ramo, sucursal) pueden introducir sesgo si están desbalanceadas en producción.
4. **Alcance**: El sistema genera **alertas de revisión**, NO acusaciones de fraude ni decisiones automáticas de pago o rechazo.
5. **Privacidad**: En producción, todos los datos de asegurados deben ser anonimizados según regulaciones locales.

> 🔒 La decisión final siempre corresponde al analista humano especializado.